In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJ = '/content/drive/MyDrive/TAC_459/'
DATA_PATH = PROJ + 'all_tickets_processed_improved_v3.csv'

In [ ]:
!pip install --quiet transformers

## Imports & Config

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

MAX_LEN   = 128
BATCH_SIZE = 32
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Load & Explore Dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
print('Nulls:', df.isnull().sum().to_dict())
print()
print('Category distribution:')
print(df['Topic_group'].value_counts().to_string())

## Label Encoding

LabelEncoder sorts categories alphabetically, giving a deterministic int→string mapping
that we can save to JSON and reload at inference time in the Streamlit app.

In [ ]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['Topic_group'])

label_names = le.classes_.tolist()   # list of string names, index = int label
num_classes = len(label_names)

print(f'Number of classes: {num_classes}')
for i, name in enumerate(label_names):
    print(f'  {i}: {name}')

## Train / Val / Test Split

In [ ]:
X = df['Document'].tolist()
y = df['label'].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42, stratify=y_train
)

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

## Class Weights

The dataset is moderately imbalanced (Hardware ~29%, Administrative rights ~3.7%).
Inverse-frequency weights passed to CrossEntropyLoss prevent the model from just
predicting the majority class.

In [ ]:
from collections import Counter

counts = Counter(y_train)
total  = sum(counts.values())
weights = [total / (num_classes * counts[i]) for i in range(num_classes)]
class_weights = torch.tensor(weights, dtype=torch.float).to(device)

print('Class weights:')
for i, (name, w) in enumerate(zip(label_names, weights)):
    print(f'  {name}: {w:.4f}  (n={counts[i]})')

## Dataset & DataLoaders

In [ ]:
class TicketDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = TicketDataset(X_train, y_train, tokenizer, MAX_LEN)
val_dataset   = TicketDataset(X_val,   y_val,   tokenizer, MAX_LEN)
test_dataset  = TicketDataset(X_test,  y_test,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE)

print(f'Batches — Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

## Model: BertCategoryClassifier

Same architecture pattern as the urgency regressor — BERT base + dropout + a single
linear head — but the head now outputs `num_classes` raw logits instead of a scalar.
CrossEntropyLoss expects raw logits (no softmax here).

In [ ]:
class BertCategoryClassifier(nn.Module):
    def __init__(self, num_classes, dropout=0.3, freeze_bert=True):
        super().__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')

        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False

        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.dropout(outputs.pooler_output)   # (B, 768)
        return self.classifier(cls)                 # (B, num_classes) — raw logits

# quick sanity check
_m = BertCategoryClassifier(num_classes=num_classes).to(device)
_dummy_ids  = torch.zeros(2, MAX_LEN, dtype=torch.long).to(device)
_dummy_mask = torch.ones(2, MAX_LEN, dtype=torch.long).to(device)
print('Output shape:', _m(_dummy_ids, _dummy_mask).shape)   # expect (2, num_classes)
del _m, _dummy_ids, _dummy_mask

## Training Helpers

Identical warmup-schedule helpers to the urgency notebook.

In [ ]:
def set_bert_layers_frozen(model, frozen=True):
    for param in model.bert.parameters():
        param.requires_grad = not frozen

def set_bert_top_layers_unfrozen(model, layers=[9, 10, 11]):
    for i in layers:
        for param in model.bert.encoder.layer[i].parameters():
            param.requires_grad = True

def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total   = 0
    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        preds   = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

    return total_loss / len(loader), correct / total


def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total   = 0
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            logits = model(input_ids, attention_mask)
            loss   = criterion(logits, labels)
            total_loss += loss.item()

            preds   = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    return total_loss / len(loader), correct / total

## Warmup Schedule Training

Same 3-phase schedule as the urgency model:
1. **head_only** (3 epochs, lr=2e-5): BERT fully frozen, only the classifier head trains.
2. **top_layers** (4 epochs, lr=1e-5): Top 3 BERT encoder layers + pooler unfrozen.
3. **all** (3 epochs, lr=5e-6): Full fine-tune at a very small rate.

In [ ]:
model_save_path = PROJ + 'bert_categorizer.pth'

model     = BertCategoryClassifier(num_classes=num_classes, dropout=0.3, freeze_bert=True).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

schedule = [
    (3, 'head_only',   2e-5),
    (4, 'top_layers',  1e-5),
    (3, 'all',         5e-6),
]

epoch_num = 0
for (num_epochs, mode, lr) in schedule:

    if mode == 'head_only':
        set_bert_layers_frozen(model, frozen=True)
        for param in model.bert.pooler.parameters():
            param.requires_grad = True
    elif mode == 'top_layers':
        set_bert_layers_frozen(model, frozen=True)
        set_bert_top_layers_unfrozen(model, layers=[9, 10, 11])
        for param in model.bert.pooler.parameters():
            param.requires_grad = True
    elif mode == 'all':
        set_bert_layers_frozen(model, frozen=False)

    optimizer = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=0.01
    )

    print(f"\n{'='*65}")
    print(f"Phase: {mode} | LR: {lr} | Trainable params: {count_trainable(model):,}")
    print(f"{'='*65}")

    for epoch in range(num_epochs):
        epoch_num += 1
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss,   val_acc   = eval_epoch(model, val_loader,   criterion, device)
        print(
            f"Epoch {epoch_num:02d} [{mode}] "
            f"| Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f} "
            f"| Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.4f}"
        )

torch.save(model.state_dict(), model_save_path)
print(f"\nModel saved to {model_save_path}")

## Save Label Mapping

The app needs to convert predicted int indices back to category name strings.
Save this alongside the weights so they always stay in sync.

In [ ]:
label_map = {str(i): name for i, name in enumerate(label_names)}
label_map_path = PROJ + 'bert_categorizer_labels.json'

with open(label_map_path, 'w') as f:
    json.dump(label_map, f, indent=2)

print(f'Label map saved to {label_map_path}')
print(json.dumps(label_map, indent=2))

## Test-Set Evaluation

In [ ]:
test_loss, test_acc = eval_epoch(model, test_loader, criterion, device)
print(f'Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc:.4f}')

# full per-class report
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
        all_preds.extend(logits.argmax(dim=1).cpu().tolist())
        all_labels.extend(batch['label'].tolist())

print()
print(classification_report(all_labels, all_preds, target_names=label_names))

## Inference

Quick smoke-test on a handful of examples before handing the weights to the app.

In [ ]:
def load_category_model(model_path, num_classes, device=None):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    m = BertCategoryClassifier(num_classes=num_classes, dropout=0.3, freeze_bert=False).to(device)
    m.load_state_dict(torch.load(model_path, map_location=device))
    m.eval()
    print(f'Model loaded from {model_path} on {device}')
    return m


def predict_category(text, model, tokenizer, label_names, max_len=128, device=None):
    """Returns (predicted_category_string, confidence_float)."""
    if device is None:
        device = next(model.parameters()).device
    text = ' '.join(text.strip().split())
    enc = tokenizer(
        text, truncation=True, padding='max_length',
        max_length=max_len, return_tensors='pt'
    )
    input_ids      = enc['input_ids'].to(device)
    attention_mask = enc['attention_mask'].to(device)
    with torch.no_grad():
        logits = model(input_ids, attention_mask)
    probs    = torch.softmax(logits, dim=1).squeeze()
    pred_idx = probs.argmax().item()
    return label_names[pred_idx], round(probs[pred_idx].item(), 4)

In [ ]:
# reload from disk to confirm the saved weights work end-to-end
with open(label_map_path) as f:
    loaded_label_map = json.load(f)
loaded_label_names = [loaded_label_map[str(i)] for i in range(len(loaded_label_map))]

model_loaded = load_category_model(model_save_path, num_classes=len(loaded_label_names))

samples = [
    "my laptop won't turn on after the update and I have a meeting in an hour",
    "please approve my vacation request for next week",
    "I need access to the shared drive for the new project",
    "running out of disk space on the file server",
    "need to purchase 3 additional software licences for the new hires",
    "can you set up a new user account for our intern starting Monday",
    "requesting admin rights to install development tools on my workstation",
    "the VPN keeps dropping when I try to connect from home",
]

print(f'{'Category':<25} {'Conf':>6}   Ticket')
print('-' * 80)
for s in samples:
    cat, conf = predict_category(s, model_loaded, tokenizer, loaded_label_names)
    print(f'{cat:<25} {conf:>6.4f}   {s[:60]}')